# Statistical Analysis and Insights

In [272]:
# Import necessary libraries

import pandas as pd
import numpy as np
from collections import Counter


In [273]:
# Make sure when we are pulling in the files that the facility ID still gets read as a string
# This will prevent any potential IDs with a leading zero to be removed

survey_clean = pd.read_csv("Data/survey_df_clean.csv", dtype = {"Facility ID": str})
hai_clean = pd.read_csv("Data/ha_df_clean.csv", dtype = {"Facility ID": str})
timely_clean = pd.read_csv("Data/timely_df_clean.csv", dtype = {"Facility ID": str})

In [274]:
# Ensure the data types are similar across all data tables before aggregation
print(survey_clean.info())
print(hai_clean.info())
print(timely_clean.info())

<class 'pandas.DataFrame'>
RangeIndex: 28584 entries, 0 to 28583
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Facility ID                 28584 non-null  str  
 1   Facility Name               28584 non-null  str  
 2   HCAHPS Measure ID           28584 non-null  str  
 3   HCAHPS Question             28584 non-null  str  
 4   Patient Survey Star Rating  28584 non-null  int64
 5   Start Date                  28584 non-null  str  
 6   End Date                    28584 non-null  str  
dtypes: int64(1), str(6)
memory usage: 4.4 MB
None
<class 'pandas.DataFrame'>
RangeIndex: 54327 entries, 0 to 54326
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Facility ID           54327 non-null  str    
 1   Facility Name         54327 non-null  str    
 2   Measure ID            54327 non-null  str    
 3   

In [275]:
# Change the title of the survey_clean measure id column to alignment
survey_clean = survey_clean.rename(columns = {"HCAHPS Measure ID": "Measure ID"})

# Double check the columns were properly renamed
print("Updated survey_clean Columns: ", survey_clean.columns)

Updated survey_clean Columns:  Index(['Facility ID', 'Facility Name', 'Measure ID', 'HCAHPS Question',
       'Patient Survey Star Rating', 'Start Date', 'End Date'],
      dtype='str')


In [276]:
# Double check to make sure there are no duplicates prior to aggregation

# Check whether each facility-measure combination is unique.
survey_duplicates = survey_clean.duplicated(
    subset=["Facility ID", "Measure ID"],
    keep=False).sum()

hai_duplicates = hai_clean.duplicated(
    subset=["Facility ID", "Measure ID"],
    keep=False).sum()

timely_duplicates = timely_clean.duplicated(
    subset=["Facility ID", "Measure ID"],
    keep=False).sum()

# For formatting
line = '--' * 40

print("Total # of Duplicated Facility-Measure Rows Within Each Dataset")
print(line)
print("Survey duplicate facility-measures:", survey_duplicates)
print("HAI duplicate facility-measures:", hai_duplicates)
print("Timely duplicate facility-measures:", timely_duplicates)

Total # of Duplicated Facility-Measure Rows Within Each Dataset
--------------------------------------------------------------------------------
Survey duplicate facility-measures: 0
HAI duplicate facility-measures: 0
Timely duplicate facility-measures: 0


In [277]:
# Ensure proper alignment of all datasets
all_df = [survey_clean, hai_clean, timely_clean]

# Create an empty dictionary to store the facilities
facilities_list = []

for df in all_df:
    if df is survey_clean:
        df['Start Date'] = pd.to_datetime(df['Start Date'], errors = "coerce")     
        df['End Date'] = pd.to_datetime(df['End Date'], errors = "coerce")

        survey_facility_list = []
        for facility in df['Facility ID']:
            survey_facility_list.append(facility)
        survey_duration = df[['Start Date', 'End Date']].value_counts()           

    elif df is hai_clean: 
        hai_facility_list = []
        for facility in df['Facility ID']:
            hai_facility_list.append(facility)
        hai_duration = df[['Start Date', 'End Date']].value_counts()           

    else:
        timely_facility_list = []
        for facility in df['Facility ID']:
            timely_facility_list.append(facility)
        timely_duration = df[['Start Date', 'End Date']].value_counts()           

print(line)
print("Unique Facilities per Data Frame: \n")
print("Number of Unique Facilities in the Survey Dataset: ", len(set(survey_facility_list)))          
print("Number of Unique Facilities in the HAI Dataset: ", len(set(hai_facility_list)))          
print("Number of Unique Facilities in the Timely Dataset: ", len(set(timely_facility_list)))      
print(line)    

print("Date Range per Data Frame: \n")
print("Unique Date Ranges for Survey Dataset: ", survey_duration)
print("Unique Date Ranges for HAI Dataset: ", hai_duration)
# print("Unique Date Ranges for Timely Dataset: ", timely_duration)

# Make sure the Start Date and End Date columns are date data types
# Create a list of dates you want to drop
dates_to_drop = ['01/01/2024', '10/01/2024']

# Create an old copy just in case
timely_clean_all_dates = timely_clean.copy()

# Keep rows where the Start Date is NOT in that list
timely_clean = timely_clean[~timely_clean['Start Date'].isin(dates_to_drop)]

print("Unique Date Ranges for Timely Dataset: ", timely_clean[['Start Date', 'End Date']].value_counts())
print(line)

--------------------------------------------------------------------------------
Unique Facilities per Data Frame: 

Number of Unique Facilities in the Survey Dataset:  3176
Number of Unique Facilities in the HAI Dataset:  4276
Number of Unique Facilities in the Timely Dataset:  4443
--------------------------------------------------------------------------------
Date Range per Data Frame: 

Unique Date Ranges for Survey Dataset:  Start Date  End Date  
2024-07-01  2025-06-30    28584
Name: count, dtype: int64
Unique Date Ranges for HAI Dataset:  Start Date  End Date  
07/01/2024  06/30/2025    54327
Name: count, dtype: int64
Unique Date Ranges for Timely Dataset:  Start Date  End Date  
07/01/2024  06/30/2025    28952
Name: count, dtype: int64
--------------------------------------------------------------------------------


In [278]:
# Compile the facilities present across all three dataset

# Create sets containing unique Facility IDs
survey_facility_list = set(survey_clean["Facility ID"])
hai_facility_list = set(hai_clean["Facility ID"])
timely_facility_list = set(timely_clean["Facility ID"])

# Combine the sets
common_facilities = survey_facility_list & hai_facility_list & timely_facility_list

# Set the data type of common_facilities to a list
common_facilities = list(common_facilities)

# Print the number of common facilities
print("Total # of Facilities Present Across All Datasets: ", len(common_facilities))

Total # of Facilities Present Across All Datasets:  2957


In [279]:
# Remove the facilities that are not present across all three datasets
survey_df = survey_clean[survey_clean['Facility ID'].isin(common_facilities)]
hai_df = hai_clean[hai_clean['Facility ID'].isin(common_facilities)]
timely_df = timely_clean[timely_clean['Facility ID'].isin(common_facilities)]

# Double check that the df have the same number of unique facilities
print(line)
print("Number of Unique Facilities in survey_df: ", len(set(survey_df['Facility ID'])))
print("Number of Unique Facilities in hai_df: ", len(set(hai_df['Facility ID'])))
print("Number of Unique Facilities in timely_df: ", len(set(timely_df['Facility ID'])))
print(line)

--------------------------------------------------------------------------------
Number of Unique Facilities in survey_df:  2957
Number of Unique Facilities in hai_df:  2957
Number of Unique Facilities in timely_df:  2957
--------------------------------------------------------------------------------


In [280]:
# Check to see if each facility has the same number of measures each
survey_measures = survey_df.groupby("Facility ID", as_index = False)['Measure ID'].nunique().rename(columns = {"Measure ID": "Measure Count"})
hai_measures = hai_df.groupby("Facility ID", as_index = False)['Measure ID'].nunique().rename(columns = {"Measure ID": "Measure Count"})
timely_measures = timely_df.groupby("Facility ID", as_index = False)['Measure ID'].nunique().rename(columns = {"Measure ID": "Measure Count"})

print(line)
print("Minimum # of Measures Available per Facility in survey_df:", min(survey_measures['Measure Count']))
print("Maximum # of Measures Available per Facility in survey_df:", max(survey_measures['Measure Count']))

print(line)
print("Minimum # of Measures Available per Facility in hai_df:", min(hai_measures['Measure Count']))
print("Maximum # of Measures Available per Facility in hai_df: ", max(hai_measures['Measure Count']))

print(line)
print("Minimum # of Measures Available per Facility in timely_df:", min(timely_measures['Measure Count']))
print("Maximum # of Measures Available per Facility in timely_df: ", max(timely_measures['Measure Count']))
print(line)

# Only keep facilities that have at least 80% of the total measures recorded
# No need to filter the survey_df since all facilities have nine total measures recorded
# HAI needs at least 15 out of 18 measures recorded
# Timely needs at least 8 out of 10 measures recorded

hai_measures = hai_measures[hai_measures['Measure Count'] >= 15]
timely_measures = timely_measures[timely_measures['Measure Count'] >= 8]

print("HAI Dataset: Total # of Facilities with at least 80% Measures Recorded: ", len(hai_measures['Facility ID']))
print("Timely Dataset: Total # of Facilities with at least 80% Measures Recorded: ", len(timely_measures['Facility ID']))
print(line)

# Update the dataframes to remove the facilities without sufficient measures
hai_df = hai_df[hai_df["Facility ID"].isin(hai_measures["Facility ID"])].copy()
timely_df = timely_df[timely_df["Facility ID"].isin(timely_measures["Facility ID"])].copy()

# Update common facilities
common_facilities = (set(survey_df["Facility ID"]) & set(hai_df["Facility ID"]) & set(timely_df["Facility ID"]))

# Print the number of facilities left
print("# of Common Facilities Remaining After Removal of Insuffiently Measured Facilities: ", len(common_facilities))
print(line)

# Update the survey_df 
survey_df = survey_df[survey_df['Facility ID'].isin(common_facilities)].reset_index(drop = True)
hai_df = hai_df[hai_df['Facility ID'].isin(common_facilities)].reset_index(drop = True)
timely_df = timely_df[timely_df['Facility ID'].isin(common_facilities)].reset_index(drop = True)

# Double check the number of facilities in each dataset -- should be the same across
print("Facilities in survey_df:", survey_df['Facility ID'].nunique())
print("Facilities in hai_df:", hai_df['Facility ID'].nunique())
print("Facilities in timely_df:", timely_df['Facility ID'].nunique())
print(line)

--------------------------------------------------------------------------------
Minimum # of Measures Available per Facility in survey_df: 9
Maximum # of Measures Available per Facility in survey_df: 9
--------------------------------------------------------------------------------
Minimum # of Measures Available per Facility in hai_df: 2
Maximum # of Measures Available per Facility in hai_df:  18
--------------------------------------------------------------------------------
Minimum # of Measures Available per Facility in timely_df: 1
Maximum # of Measures Available per Facility in timely_df:  10
--------------------------------------------------------------------------------
HAI Dataset: Total # of Facilities with at least 80% Measures Recorded:  1880
Timely Dataset: Total # of Facilities with at least 80% Measures Recorded:  2294
--------------------------------------------------------------------------------
# of Common Facilities Remaining After Removal of Insuffiently Measured 

# Statistical Analysis: Survey

In [281]:
# As a reminder, print the columns of the survey_df
print(line)
print("Survey Data Frame Columns: ", survey_df.columns.tolist())
print(line)

# Pull the unique measures
print("Survey Measures: ")
print(survey_df[['Measure ID', 'HCAHPS Question']].value_counts())
print(line)

--------------------------------------------------------------------------------
Survey Data Frame Columns:  ['Facility ID', 'Facility Name', 'Measure ID', 'HCAHPS Question', 'Patient Survey Star Rating', 'Start Date', 'End Date']
--------------------------------------------------------------------------------
Survey Measures: 
Measure ID                HCAHPS Question                            
H_COMP_1_STAR_RATING      Nurse communication - star rating              1646
H_COMP_2_STAR_RATING      Doctor communication - star rating             1646
H_COMP_5_STAR_RATING      Communication about medicines - star rating    1646
H_COMP_6_STAR_RATING      Discharge information - star rating            1646
H_CLEAN_STAR_RATING       Cleanliness - star rating                      1646
H_QUIET_STAR_RATING       Quietness - star rating                        1646
H_HSP_RATING_STAR_RATING  Overall hospital rating - star rating          1646
H_RECMND_STAR_RATING      Recommend hospital - star ra

In [282]:
# Assess new shape of survey df -- should be 1646 facilities * 9 measures = 14814 rows and 7 columns
print("Final Shape of survey_df: ", survey_df.shape) 

Final Shape of survey_df:  (14814, 7)


In [283]:
# Distribution of Star Ratings
rating_counts = survey_df["Patient Survey Star Rating"].value_counts().sort_index()

# Generate the proportion of rating to grand total
rating_percent = (survey_df["Patient Survey Star Rating"].value_counts(normalize=True).sort_index() * 100).round(2)

# Place the metrics into df
rating_summary = pd.DataFrame({
    "Count": rating_counts,
    "Percent %": rating_percent})

print(line)
print("Count and Proportion of Star Ratings Reported: \n")
print(rating_summary)
print(line)

# Generate the average rating for each measure
measure_average = (survey_df.groupby("HCAHPS Question")["Patient Survey Star Rating"].mean().round(2).sort_values(ascending = False))

print("Average Rating per Measure: \n")
print(measure_average)
print(line)

# Generate the descriptive statistics for the df
measure_stats = (survey_df.groupby("HCAHPS Question")["Patient Survey Star Rating"].agg(["mean", "median", "std", "min", "max"]).round(2).sort_values(by = 'mean', ascending = False))

print("Descriptive Statistics for survey_df: \n")
print(measure_stats)
print(line)

--------------------------------------------------------------------------------
Count and Proportion of Star Ratings Reported: 

                            Count  Percent %
Patient Survey Star Rating                  
1                             706       4.77
2                            3859      26.05
3                            6282      42.41
4                            3516      23.73
5                             451       3.04
--------------------------------------------------------------------------------
Average Rating per Measure: 

HCAHPS Question
Recommend hospital - star rating               3.41
Discharge information - star rating            3.10
Nurse communication - star rating              3.03
Cleanliness - star rating                      3.00
Overall hospital rating - star rating          2.99
Doctor communication - star rating             2.97
Summary star rating                            2.96
Quietness - star rating                        2.78
Communicatio

In [284]:
# Generate descriptive analysis on aggregated data for facility-level insight
survey_summary = (survey_df
    .groupby(["Facility ID", "Facility Name"])
    .agg(
        Measures_Captured = ("Measure ID", "nunique"),
        Average_Rating = ("Patient Survey Star Rating", "mean"),
        Median_Rating = ("Patient Survey Star Rating", "median"),
        Minimum_Rating = ("Patient Survey Star Rating", "min"),
        Maximum_Rating = ("Patient Survey Star Rating", "max"),
        Rating_SD = ("Patient Survey Star Rating", "std"))
    .round(2).sort_values(by = 'Facility ID', ascending = True).reset_index())

# Print out the summary
survey_summary.head(5)

,Facility ID,Facility Name,Measures_Captured,Average_Rating,Median_Rating,Minimum_Rating,Maximum_Rating,Rating_SD
0,010005,MARSHALL MEDICAL CENTERS,9,3.11,3.0,2,4,0.60
1,010006,NORTH ALABAMA MEDICAL CENTER,9,2.11,2.0,1,4,0.78
2,010011,ST. VINCENT'S EAST,9,2.78,3.0,2,3,0.44
3,010019,HELEN KELLER HOSPITAL,9,3.78,4.0,3,5,0.67
4,010023,BAPTIST MEDICAL CENTER SOUTH,9,3.00,3.0,3,3,0.00


In [285]:
# Generate a matrix for facility and measures

# Remove "- star rating" suffix from HCAHPS Question
survey_df['HCAHPS Question'] = survey_df['HCAHPS Question'].str.partition(' -')[0]

# Create pivot table with measures as columns
survey_matrix = survey_df.pivot_table(
    index=["Facility ID", "Facility Name"],
    columns="HCAHPS Question",
    values="Patient Survey Star Rating")

# Print to double check
survey_matrix.head(10)

,HCAHPS Question,Cleanliness,Communication about medicines,Discharge information,Doctor communication,Nurse communication,Overall hospital rating,Quietness,Recommend hospital,Summary star rating
Facility ID,Facility Name,,,,,,,,,
010005,MARSHALL MEDICAL CENTERS,3.0,2.0,3.0,4.0,3.0,3.0,4.0,3.0,3.0
010006,NORTH ALABAMA MEDICAL CENTER,2.0,1.0,2.0,2.0,2.0,2.0,4.0,2.0,2.0
010011,ST. VINCENT'S EAST,2.0,2.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
010019,HELEN KELLER HOSPITAL,3.0,3.0,4.0,3.0,4.0,4.0,4.0,5.0,4.0
010023,BAPTIST MEDICAL CENTER SOUTH,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
010024,JACKSON HOSPITAL & CLINIC INC,1.0,2.0,2.0,3.0,3.0,3.0,4.0,3.0,3.0
010029,THE EAST ALABAMA HEALTHCARE AUTHORITY,3.0,3.0,3.0,4.0,4.0,4.0,4.0,4.0,4.0
010033,UNIVERSITY OF ALABAMA HOSPITAL,2.0,3.0,4.0,4.0,3.0,4.0,3.0,5.0,4.0
010035,CULLMAN REGIONAL MEDICAL CENTER,3.0,2.0,3.0,3.0,4.0,3.0,4.0,3.0,3.0


In [286]:
# Figure out the best and worst performing measure per facility
survey_best_worst = survey_df.pivot_table(
    index=["Facility ID", "Facility Name"],
    columns="HCAHPS Question",
    values="Patient Survey Star Rating")

# Select only the numeric rating columns
survey_numerical_columns = survey_best_worst.select_dtypes(include="number").columns

# Create Best and Worst Performing Measure
survey_best_worst["Best_Performing_Measure"] = survey_best_worst[survey_numerical_columns].idxmax(axis=1)
survey_best_worst["Worst_Performing_Measure"] = survey_best_worst[survey_numerical_columns].idxmin(axis=1)

# Only keep the best and worst measure columns, other rating columns can be called from the matrix
survey_best_worst = survey_best_worst.drop(columns = ['Cleanliness', 'Communication about medicines', 'Discharge information', 'Nurse communication', 'Doctor communication', 'Overall hospital rating', 'Quietness', 'Recommend hospital', 'Summary star rating'])

# Double check output
survey_best_worst.head()

,HCAHPS Question,Best_Performing_Measure,Worst_Performing_Measure
Facility ID,Facility Name,,
010005,MARSHALL MEDICAL CENTERS,Doctor communication,Communication about medicines
010006,NORTH ALABAMA MEDICAL CENTER,Quietness,Communication about medicines
010011,ST. VINCENT'S EAST,Discharge information,Cleanliness
010019,HELEN KELLER HOSPITAL,Recommend hospital,Cleanliness
010023,BAPTIST MEDICAL CENTER SOUTH,Cleanliness,Cleanliness


In [287]:
# Compare facility performance to performance of all applicable facilities in dataset
overall_average = (survey_df.groupby("HCAHPS Question")["Patient Survey Star Rating"].mean())

# Double check the overall average is pulled
print(line)
print(overall_average)
print(line)

# Calculate the difference between the overall average and facility score
survey_comparison = survey_matrix - overall_average

# Print the survey_comparison
survey_comparison.head(5)


--------------------------------------------------------------------------------
HCAHPS Question
Cleanliness                      3.001215
Communication about medicines    2.247874
Discharge information            3.095990
Doctor communication             2.969016
Nurse communication              3.027947
Overall hospital rating          2.989064
Quietness                        2.777035
Recommend hospital               3.411300
Summary star rating              2.962333
Name: Patient Survey Star Rating, dtype: float64
--------------------------------------------------------------------------------


,HCAHPS Question,Cleanliness,Communication about medicines,Discharge information,Doctor communication,Nurse communication,Overall hospital rating,Quietness,Recommend hospital,Summary star rating
Facility ID,Facility Name,,,,,,,,,
010005,MARSHALL MEDICAL CENTERS,-0.001215,-0.247874,-0.09599,1.030984,-0.027947,0.010936,1.222965,-0.4113,0.037667
010006,NORTH ALABAMA MEDICAL CENTER,-1.001215,-1.247874,-1.09599,-0.969016,-1.027947,-0.989064,1.222965,-1.4113,-0.962333
010011,ST. VINCENT'S EAST,-1.001215,-0.247874,-0.09599,0.030984,-0.027947,0.010936,0.222965,-0.4113,0.037667
010019,HELEN KELLER HOSPITAL,-0.001215,0.752126,0.90401,0.030984,0.972053,1.010936,1.222965,1.5887,1.037667
010023,BAPTIST MEDICAL CENTER SOUTH,-0.001215,0.752126,-0.09599,0.030984,-0.027947,0.010936,0.222965,-0.4113,0.037667


In [288]:
# Since the values are difficult to interpret, categorize the values using "Below Average", "Average", "Above Average"
survey_categories = survey_comparison.apply(lambda column: pd.cut(column, bins = [-np.inf, -.5, .5, np.inf], labels = ["Below Average", "Average", "Above Average"], include_lowest = True))

# Double check it was successful
survey_categories.head(5)

,HCAHPS Question,Cleanliness,Communication about medicines,Discharge information,Doctor communication,Nurse communication,Overall hospital rating,Quietness,Recommend hospital,Summary star rating
Facility ID,Facility Name,,,,,,,,,
010005,MARSHALL MEDICAL CENTERS,Average,Average,Average,Above Average,Average,Average,Above Average,Average,Average
010006,NORTH ALABAMA MEDICAL CENTER,Below Average,Below Average,Below Average,Below Average,Below Average,Below Average,Above Average,Below Average,Below Average
010011,ST. VINCENT'S EAST,Below Average,Average,Average,Average,Average,Average,Average,Average,Average
010019,HELEN KELLER HOSPITAL,Average,Above Average,Above Average,Average,Above Average,Above Average,Above Average,Above Average,Above Average
010023,BAPTIST MEDICAL CENTER SOUTH,Average,Above Average,Average,Average,Average,Average,Average,Average,Average


In [289]:
# Create a dictionary to store the facility-level insights

# First merge all the insights   
merged_survey_insights = survey_summary.merge(survey_best_worst, on = ["Facility ID", "Facility Name"]).merge(survey_categories, on = ["Facility ID", "Facility Name"]).merge(survey_matrix, on = ["Facility ID", "Facility Name"])

# Convert survey_insights to dictionary
survey_insights = merged_survey_insights.set_index("Facility ID").to_dict(orient = "index")

# Create a for loop to test out a tester
for facility_id, info in survey_insights.items():

    # List out the values present in survey_insight
    print(info)

    print("==" * 40)
    print(f"Facility ID: {facility_id}")
    print(f"Facility Name: {info['Facility Name']}")
    print()


    print("Patient Star Rating for Each Measure")
    print(line)
    print(f"Cleanliness Rating: {info['Cleanliness_y']}")
    print(f"Communication About Medication Rating: {info['Communication about medicines_y']}")
    print(f"Discharge Information Rating: {info['Discharge information_y']}")
    print(f"Doctor Communication Rating: {info['Doctor communication_y']}")
    print(f"Nurse Communication Rating: {info['Nurse communication_y']}")
    print(f"Overall Hospital Rating: {info['Overall hospital rating_y']}")
    print(f"Quietness Rating: {info['Quietness_y']}")
    print(f"Recommend Hospital Rating: {info['Recommend hospital_y']}")
    print(f"Summary Star Rating Rating: {info['Summary star rating_y']}")
    print()

    print("Descriptive Statistics for Each Facility")
    print(line)
    print(f"Measures Captured : {info['Measures_Captured']}")
    print(f"Average Rating : {info['Average_Rating']}")
    print(f"Median Rating  : {info['Median_Rating']}")
    print(f"Minimum Rating  : {info['Minimum_Rating']}")
    print(f"Maximum Rating  : {info['Maximum_Rating']}")
    print(f"Median Rating  : {info['Median_Rating']}")
    print(f"Rating SD      : {info['Rating_SD']}")   
    print()

    print("Best and Worst Performing Measures for Each Facility")
    print(line)
    print(f"Best Measure  : {info['Best_Performing_Measure']}")
    print(f"Worst Measure : {info['Worst_Performing_Measure']}")
    print()

    print("Overall Facility Performance Compared to Overall National Average Performance")
    print(line)
    print(f"Overall Hospital Rating : {info['Overall hospital rating_x']}")
    print(f"Summary star rating : {info['Summary star rating_x']}")
    print("==" * 40)

    break

{'Facility Name': 'MARSHALL MEDICAL CENTERS', 'Measures_Captured': 9, 'Average_Rating': 3.11, 'Median_Rating': 3.0, 'Minimum_Rating': 2, 'Maximum_Rating': 4, 'Rating_SD': 0.6, 'Best_Performing_Measure': 'Doctor communication', 'Worst_Performing_Measure': 'Communication about medicines', 'Cleanliness_x': 'Average', 'Communication about medicines_x': 'Average', 'Discharge information_x': 'Average', 'Doctor communication_x': 'Above Average', 'Nurse communication_x': 'Average', 'Overall hospital rating_x': 'Average', 'Quietness_x': 'Above Average', 'Recommend hospital_x': 'Average', 'Summary star rating_x': 'Average', 'Cleanliness_y': 3.0, 'Communication about medicines_y': 2.0, 'Discharge information_y': 3.0, 'Doctor communication_y': 4.0, 'Nurse communication_y': 3.0, 'Overall hospital rating_y': 3.0, 'Quietness_y': 4.0, 'Recommend hospital_y': 3.0, 'Summary star rating_y': 3.0}
Facility ID: 010005
Facility Name: MARSHALL MEDICAL CENTERS

Patient Star Rating for Each Measure
------------

In [290]:
# Double check to see if all metrics are present for each facility
print(list(survey_insights.values())[:2])

[{'Facility Name': 'MARSHALL MEDICAL CENTERS', 'Measures_Captured': 9, 'Average_Rating': 3.11, 'Median_Rating': 3.0, 'Minimum_Rating': 2, 'Maximum_Rating': 4, 'Rating_SD': 0.6, 'Best_Performing_Measure': 'Doctor communication', 'Worst_Performing_Measure': 'Communication about medicines', 'Cleanliness_x': 'Average', 'Communication about medicines_x': 'Average', 'Discharge information_x': 'Average', 'Doctor communication_x': 'Above Average', 'Nurse communication_x': 'Average', 'Overall hospital rating_x': 'Average', 'Quietness_x': 'Above Average', 'Recommend hospital_x': 'Average', 'Summary star rating_x': 'Average', 'Cleanliness_y': 3.0, 'Communication about medicines_y': 2.0, 'Discharge information_y': 3.0, 'Doctor communication_y': 4.0, 'Nurse communication_y': 3.0, 'Overall hospital rating_y': 3.0, 'Quietness_y': 4.0, 'Recommend hospital_y': 3.0, 'Summary star rating_y': 3.0}, {'Facility Name': 'NORTH ALABAMA MEDICAL CENTER', 'Measures_Captured': 9, 'Average_Rating': 2.11, 'Median_Ra

In [291]:
clean_survey_insights = {}

for facility_id, info in survey_insights.items():

    clean_survey_insights[facility_id] = {
        "Facility Name": info["Facility Name"],
        "Facility ID": facility_id,

        "Summary Statistics": {
            "Measures Captured": info["Measures_Captured"],
            "Average Rating": info["Average_Rating"],
            "Median Rating": info["Median_Rating"],
            "Minimum Rating": info["Minimum_Rating"],
            "Maximum Rating": info["Maximum_Rating"],
            "Rating Standard Deviation": info["Rating_SD"]},

        "Performance Highlights": {
            "Best Performing Measure": info["Best_Performing_Measure"],
            "Worst Performing Measure": info["Worst_Performing_Measure"]},

        "Measure Results": {
            "Cleanliness": {
                "Rating": info["Cleanliness_y"],
                "Performance": info["Cleanliness_x"]},

            "Communication About Medicines": {
                "Rating": info["Communication about medicines_y"],
                "Performance": info["Communication about medicines_x"]},

            "Discharge Information": {
                "Rating": info["Discharge information_y"],
                "Performance": info["Discharge information_x"]},

            "Doctor Communication": {
                "Rating": info["Doctor communication_y"],
                "Performance": info["Doctor communication_x"]},

            "Nurse Communication": {
                "Rating": info["Nurse communication_y"],
                "Performance": info["Nurse communication_x"]},

            "Overall Hospital Rating": {
                "Rating": info["Overall hospital rating_y"],
                "Performance": info["Overall hospital rating_x"]},

            "Quietness": {
                "Rating": info["Quietness_y"],
                "Performance": info["Quietness_x"]},

            "Recommend Hospital": {
                "Rating": info["Recommend hospital_y"],
                "Performance": info["Recommend hospital_x"]},

            "Summary Star Rating": {
                "Rating": info["Summary star rating_y"],
                "Performance": info["Summary star rating_x"]}
        }
    }

In [292]:
# Store the survey categories
survey_categories = list(clean_survey_insights[facility_id].keys())

# Print the list to double check 
print(line)
print("Survey Categories: ", survey_categories)

# Store the measures by calling a tester facility

# Set an example facility to pull the measures
test_facility = clean_survey_insights["010005"]

# Place the measures into a list
survey_measures = list(test_facility["Measure Results"].keys())

# Print to double check
print("Survey Measures: ", survey_measures)

# Store the measurement types
survey_measurements = []

for measure in test_facility["Measure Results"].keys():
    for measurement in test_facility["Measure Results"][measure]:
        survey_measurements.append(measurement)
    break

print("Survey Measurement Types: ", survey_measurements)
print(line)

--------------------------------------------------------------------------------
Survey Categories:  ['Facility Name', 'Facility ID', 'Summary Statistics', 'Performance Highlights', 'Measure Results']
Survey Measures:  ['Cleanliness', 'Communication About Medicines', 'Discharge Information', 'Doctor Communication', 'Nurse Communication', 'Overall Hospital Rating', 'Quietness', 'Recommend Hospital', 'Summary Star Rating']
Survey Measurement Types:  ['Rating', 'Performance']
--------------------------------------------------------------------------------


In [293]:
# Test to see how to call metrics
# Format: clean_survey_insights["facility_id"]["survey_category"]["Measure"]["Measurement Type"]
print(line)
print("Facility Name: ", test_facility["Facility Name"])
print("Facility ID: ", test_facility["Facility ID"])

print("Average Rating from Facility Summary Statistics: ", test_facility["Summary Statistics"]["Average Rating"])

print("Best Performing Measure at Facility: ", test_facility["Performance Highlights"]["Best Performing Measure"])

print("Doctor Communication Rating at Facility: ", test_facility["Measure Results"]["Doctor Communication"]["Rating"])

print("Facility Performance of Doctor Communication Compared to National Average: ", test_facility["Measure Results"]["Doctor Communication"]["Performance"])
print(line)

--------------------------------------------------------------------------------
Facility Name:  MARSHALL MEDICAL CENTERS
Facility ID:  010005
Average Rating from Facility Summary Statistics:  3.11
Best Performing Measure at Facility:  Doctor communication
Doctor Communication Rating at Facility:  4.0
Facility Performance of Doctor Communication Compared to National Average:  Above Average
--------------------------------------------------------------------------------


In [294]:
# Create a survey document with every metric being a sentence
# This will allow for summary generation down the line

def create_survey_document(facility_id, facility_info):

    # Pull the major dictionary sections, start with summary statistics
    summary = facility_info["Summary Statistics"]

    # Pull the performance highlights next
    highlights = facility_info["Performance Highlights"]

    # Finally pull the results from the measures
    measure_results = facility_info["Measure Results"]

    # Create empty list to store one readable sentence for every measure 
    measure_sentences = []

    # Create a for loop to iterate through every measure
    for measure_name, result in measure_results.items():

        # Indicate the type of measurement
        rating = result["Rating"]
        performance = result["Performance"]

        # Define the structure of the sentence
        sentence = (f"The {measure_name} measure was given a {rating} star rating and was categorized as {performance.lower()} compared with the aggregated average across all facilities.")

        # Append the sentence to the emplty measure_sentence
        measure_sentences.append(sentence)

    # Combine all measure sentences together
    measure_text = " ".join(measure_sentences)

    # Create the complete facility document
    document = (

        f"{facility_info['Facility Name']} (Facility ID: #{facility_id}) captured {summary['Measures Captured']} HCAHPS patient survey measures. "

        f"The average star rating was {summary['Average Rating']}, and the median rating was {summary['Median Rating']}. "

        f"Ratings ranged from {summary['Minimum Rating']} to {summary['Maximum Rating']}. "

        f"The standard deviation of the ratings was {summary['Rating Standard Deviation']}. "

        f"The best-performing measure was {highlights['Best Performing Measure']}. "

        f"The worst-performing measure was {highlights['Worst Performing Measure']}. "

        f"Individual measure results: {measure_text}")

    return document

In [295]:
# Create an empty list to store survey documents
survey_documents = []

# Create a for loop to create a document for each of the facilities
for facility_id, facility_info in clean_survey_insights.items():

    # Call the create_survey_document function and store it under document_text
    document_text = create_survey_document(facility_id = facility_id, facility_info = facility_info)

    # Now append the document text to the empty list previously created to store the documents
    survey_documents.append({
        "Document ID": (f"survey_{facility_id}"),
        "Facility ID": facility_id,
        "Facility Name": facility_info["Facility Name"],
        "Dataset": "HCAHPS Patient Survey",
        "Survey Report": document_text})

# The total number of survey documents should be equal to the unique facilities (1646)
print("Number of survey documents created:", len(survey_documents))

Number of survey documents created: 1646


In [296]:
# Inspect the first document
test_document = survey_documents[0]

print(line)
print(test_document["Survey Report"])
print(line)

# Convert the documents into a dataframe
survey_documents_df = pd.DataFrame(survey_documents)

# Check the columns and the survey report
print("Columns in survey_documents_df: ", survey_documents_df.columns.tolist())
print(line)

--------------------------------------------------------------------------------
MARSHALL MEDICAL CENTERS (Facility ID: #010005) captured 9 HCAHPS patient survey measures. The average star rating was 3.11, and the median rating was 3.0. Ratings ranged from 2 to 4. The standard deviation of the ratings was 0.6. The best-performing measure was Doctor communication. The worst-performing measure was Communication about medicines. Individual measure results: The Cleanliness measure was given a 3.0 star rating and was categorized as average compared with the aggregated average across all facilities. The Communication About Medicines measure was given a 2.0 star rating and was categorized as average compared with the aggregated average across all facilities. The Discharge Information measure was given a 3.0 star rating and was categorized as average compared with the aggregated average across all facilities. The Doctor Communication measure was given a 4.0 star rating and was categorized as a

In [ ]:
# Store the reporting data as the metadata for each facility
survey_documents_df['Metadata'] = [
    {
        "File Name ": "HCAHPS-Hospital.csv",
        "Regulatory Body": "Centers for Medicare & Medicaid Services (CMS)",
        "Source": "Hospital Consumer Assessment of Healthcare Providers and Systems",
        "Website": "Care Compare Website"
        "Link": "https://hcahpsonline.org/en/hcahps-star-ratings",
        "Reporting Date": {"Start Date": "7/1/2024", "End Date": "6/30/2025"}
    }] * len(survey_documents_df)

# Reminder of what the metadata column stores
# To call the metadata, use survey_documents_df.loc[0, "Metadata"]["Start Date"]
survey_metadata = ['Start Date', 'End Date']

print(line)
print("Metadata within Survey Data Frame Contains the Following: ", survey_metadata)
print(line)

# Double check the storage of survey_documents_df
print(survey_documents_df.head(2))
print(line)

--------------------------------------------------------------------------------
Metadata within Survey Data Frame Contains the Following:  ['Start Date', 'End Date']
--------------------------------------------------------------------------------
     Document ID Facility ID                 Facility Name  \
0  survey_010005      010005      MARSHALL MEDICAL CENTERS   
1  survey_010006      010006  NORTH ALABAMA MEDICAL CENTER   

                 Dataset                                      Survey Report  \
0  HCAHPS Patient Survey  MARSHALL MEDICAL CENTERS (Facility ID: #010005...   
1  HCAHPS Patient Survey  NORTH ALABAMA MEDICAL CENTER (Facility ID: #01...   

                                            Metadata  
0  {'File Name ': 'HCAHPS-Hospital.csv', 'Source'...  
1  {'File Name ': 'HCAHPS-Hospital.csv', 'Source'...  
--------------------------------------------------------------------------------


# Statistical Insight: HAI

In [309]:
# Pull in the cleaned df
print(line)
print("HAI Data Frame Columns: ", hai_df.columns.tolist())
print(line)

# Make sure the number of unique facilities matches the previous df -- should be 1646
print("Total # of Unique Facilities: ", hai_df['Facility ID'].nunique())
print(line)

# Double check for null values
print("Null Values Within hai_df: ", hai_df.isnull().sum())
print(line)

# Assess how many measures exist
print("Total # of Unique Measures Captured: ", hai_df[['Measure ID', 'Measure Name']].value_counts())

--------------------------------------------------------------------------------
HAI Data Frame Columns:  ['Facility ID', 'Facility Name', 'Measure ID', 'Measure Name', 'Compared to National', 'Score', 'Start Date', 'End Date']
--------------------------------------------------------------------------------
Total # of Unique Facilities:  1646
--------------------------------------------------------------------------------
Null Values Within hai_df:  Facility ID             0
Facility Name           0
Measure ID              0
Measure Name            0
Compared to National    0
Score                   0
Start Date              0
End Date                0
dtype: int64
--------------------------------------------------------------------------------
Total # of Unique Measures Captured:  Measure ID       Measure Name                                                                                      
HAI_1_DOPC       Central Line Associated Bloodstream Infection: Number of Device Days     